# Repro Paper 1: *ViSoBERT: A Pre-trained Language Model for Vietnamese Social Media Text Processing*
**Venue:** EMNLP 2023 Main Conference (pp. 5191-5207)
**Authors:** Quoc-Nam Nguyen, Thang Chau Phan, Duc-Vu Nguyen, Kiet Van Nguyen
**Paper Link:** [ACL Anthology 2023.emnlp-main.315](https://aclanthology.org/2023.emnlp-main.315)
**Target:** Reproduce Table 4 benchmarks on the official UIT public datasets: `UIT-VSMEC` and `UIT-ViHSD`.

### Published Table 4 Targets (Macro F1 %):
| Model | UIT-VSMEC (Emotion, 7 classes) | UIT-ViHSD (Hate Speech, 3 classes) |
| :--- | :---: | :---: |
| `vinai/phobert-base` | 61.41 | 65.01 |
| `xlm-roberta-base` | 58.67 | 63.39 |
| **`uitnlp/visobert` (Ours)** | **65.88** | **68.77** |

### Dataset provenance (verified 2026-11-09 against the HF datasets-server)
| Task | Hub repository | Split sizes | Note |
| :--- | :--- | :--- | :--- |
| UIT-VSMEC | `duwuonline/UIT-VSMEC` | 5548 / 686 / 693 | Public mirror of the UIT release; columns `Sentence`, `Emotion` |
| UIT-ViHSD | `htdung167/ViHSD` | 24048 / 2672 / 6680 | Public mirror; columns `free_text`, `label_id` |

> The original `uitnlp/uit-vsmec` repository no longer resolves and `uitnlp/vihsd` is a gated
> repository (terms acceptance + token required). The mirrors above are used because their row
> counts match Table 2 of the paper exactly, which keeps the split-size assertions meaningful.
> This substitution is a documented deviation and must be disclosed in the report.

### Known Library & Implementation Deviations:
- **Original Paper:** Trained using `the simpletransformers` (ver 0.63.11 in footnote 15; default linear warmup, weight_decay = 0.0, AdamW eps = 1e-8). Procedures outlined in `(Devlin et al., 2019)`.
- **Dataset Names in Paper:** `UIT-VSMEC` (Emotion Recognition) and `UIT-ViHSD` (Hate Speech Detection).
- **This Replication:** Implemented via standard HuggingFace `Trainer` (`transformers` library).
- **Strict Anti-Data-Leakage:** Checkpoints are chosen strictly using validation (dev) Macro-F1 (`metric_for_best_model="macro_f1"`). The test set is evaluated exactly once after training, and the best-checkpoint selection never observes test labels.

In [ ]:
# Cell 1: Environment Setup, Deterministic Seeding & Version Shims
!pip install -q transformers datasets evaluate scikit-learn accelerate sentencepiece

import inspect
import json
import os
import random
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    set_seed,
)
from sklearn.metrics import f1_score, accuracy_score

def set_deterministic_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    set_seed(seed)

set_deterministic_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device} | CUDA Available: {torch.cuda.is_available()}")
print(f"transformers: {__import__('transformers').__version__}")

# --- transformers version shims (4.x vs 5.x) ---
TA_FIELDS = getattr(TrainingArguments, "__dataclass_fields__", {})
EVAL_STRATEGY_KEY = "eval_strategy" if "eval_strategy" in TA_FIELDS else "evaluation_strategy"
print(f"Eval-strategy keyword: {EVAL_STRATEGY_KEY}")

def build_trainer(**kwargs):
    params = inspect.signature(Trainer.__init__).parameters
    if "processing_class" in params:
        kwargs["processing_class"] = kwargs.pop("tokenizer")
    return Trainer(**kwargs)

In [ ]:
# Cell 2: Load Official UIT Datasets with Strict Split Assertions
# Execution order recommendation: run "UIT-VSMEC" first (fast, N_test=693), then append
# "UIT-ViHSD" (N_test=6680). Set the REPRO_TASKS env var to switch without editing code.
BENCHMARK_TASKS = [t for t in os.environ.get("REPRO_TASKS", "UIT-VSMEC").split(",") if t]

# Verified against the HF datasets-server on 2026-11-09.
DATASET_CANDIDATES = {
    "UIT-VSMEC": [
        ("duwuonline/UIT-VSMEC", "Sentence", "Emotion"),
        ("ura-hcmut/UIT-VSMEC", "Sentence", "Emotion"),
    ],
    "UIT-ViHSD": [
        ("htdung167/ViHSD", "free_text", "label_id"),
        ("SEACrowd/uit_vihsd", None, None),
    ],
}

EXPECTED_SPLITS = {
    "UIT-VSMEC": (5548, 686, 693),
    "UIT-ViHSD": (24048, 2672, 6680),
}

PUBLISHED_TABLE4 = {
    "UIT-VSMEC": {
        "vinai/phobert-base": {"MF1": 61.41, "WF1": 63.36, "Acc": 63.49},
        "xlm-roberta-base": {"MF1": 58.67, "WF1": 61.02, "Acc": 60.92},
        "uitnlp/visobert": {"MF1": 65.88, "WF1": 68.37, "Acc": 68.10},
    },
    "UIT-ViHSD": {
        "vinai/phobert-base": {"MF1": 65.01, "WF1": 86.81, "Acc": 87.12},
        "xlm-roberta-base": {"MF1": 63.39, "WF1": 86.08, "Acc": 86.36},
        "uitnlp/visobert": {"MF1": 68.77, "WF1": 88.31, "Acc": 88.51},
    },
}

def _resolve_columns(ds, text_col, label_col):
    feats = list(ds["train"].features.keys())
    if text_col is None or text_col not in feats:
        candidates = [c for c in feats if ds["train"].features[c].dtype == "string"]
        text_col = candidates[0] if candidates else None
    if label_col is None or label_col not in feats:
        for c in feats:
            if c.lower() in ("label", "label_id", "labels", "emotion", "target"):
                label_col = c
                break
    if text_col is None or label_col is None:
        raise ValueError(f"Could not resolve text/label columns from features: {feats}")
    return text_col, label_col

def load_official_uit_dataset(task_name: str):
    expected = EXPECTED_SPLITS[task_name]
    errors = []
    for repo, text_col, label_col in DATASET_CANDIDATES[task_name]:
        try:
            ds = load_dataset(repo)
        except Exception as exc:  # noqa: BLE001 - we want the reason logged, not raised yet
            errors.append(f"{repo}: {exc}")
            print(f"[skip] {repo} -> {type(exc).__name__}: {exc}")
            continue

        def pick(*names):
            for n in names:
                if n in ds:
                    return n
            return None

        tr_k, va_k, te_k = pick("train"), pick("validation", "dev"), pick("test")
        if tr_k is None or te_k is None:
            errors.append(f"{repo}: missing train/test split (found {list(ds.keys())})")
            continue

        text_col, label_col = _resolve_columns(ds, text_col, label_col)
        train_df = pd.DataFrame(ds[tr_k])
        test_df = pd.DataFrame(ds[te_k])

        deviation_note = ""
        if va_k is not None:
            val_df = pd.DataFrame(ds[va_k])
        else:
            # Honest fallback: carve a stratified dev split out of train and DISCLOSE it.
            from sklearn.model_selection import train_test_split
            train_df, val_df = train_test_split(
                train_df, test_size=0.1, random_state=42, stratify=train_df[label_col]
            )
            deviation_note = "NO-DEV-SPLIT FALLBACK: 10% stratified carve from train"

        got = (len(train_df), len(val_df), len(test_df))
        strict_ok = got == expected
        print(f"[{repo}] splits found: train={got[0]}, dev={got[1]}, test={got[2]} "
              f"(paper Table 2: {expected[0]}/{expected[1]}/{expected[2]})")
        if not strict_ok and not deviation_note:
            errors.append(f"{repo}: split mismatch {got} != {expected}")
            continue

        unique_labels = sorted(pd.concat([train_df, val_df, test_df])[label_col].unique().tolist())
        label2id = {lbl: idx for idx, lbl in enumerate(unique_labels)}
        for df in (train_df, val_df, test_df):
            df["text"] = df[text_col].astype(str)
            df["label"] = df[label_col].map(label2id).astype(int)

        print(f"[{task_name}] source={repo} text_col={text_col} label_col={label_col} "
              f"classes={len(unique_labels)} {deviation_note}")
        print(f"[{task_name}] class counts (train): "
              f"{train_df['label'].value_counts().sort_index().to_dict()}")
        assert len(test_df) == expected[2], "test split must match the paper exactly"
        return (
            train_df[["text", "label"]],
            val_df[["text", "label"]],
            test_df[["text", "label"]],
            len(unique_labels),
            repo,
        )
    raise RuntimeError(
        "No usable public source for " + task_name + ". Tried:\n  " + "\n  ".join(errors)
    )

# Probe the datasets without training so a broken pin fails fast.
for _task in BENCHMARK_TASKS:
    _tr, _va, _te, _n, _src = load_official_uit_dataset(_task)
    print(f"OK {_task}: {_src} | N_test={len(_te)} | classes={_n}")

In [ ]:
# Cell 3: Multi-Seed Replication Pipeline with Dev Checkpoint Selection
import gc
import time

MODELS = ["xlm-roberta-base", "vinai/phobert-base", "uitnlp/visobert"]
SEEDS = [42, 43, 44]
RUN_LOG = "Repro_Paper1_runs.jsonl"
LATENCY_SAMPLES = 50

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "macro_f1": f1_score(labels, preds, average="macro") * 100.0,
        "weighted_f1": f1_score(labels, preds, average="weighted") * 100.0,
        "accuracy": accuracy_score(labels, preds) * 100.0,
    }

def measure_cpu_latency(model, tokenizer, texts, max_length=128, warmup=20, reps=LATENCY_SAMPLES):
    # Measure a fp32 CPU copy so fp16 weights never hit CPU-only kernels; the GPU model
    # under evaluation stays untouched.
    import copy
    cpu_model = copy.deepcopy(model).float().to("cpu").eval()
    sample = [t[:512] for t in texts[:reps]]
    with torch.no_grad():
        for _ in range(warmup):
            _ = cpu_model(**tokenizer(sample[0], return_tensors="pt", truncation=True,
                                     max_length=max_length))
        lats = []
        for txt in sample:
            enc = tokenizer(txt, return_tensors="pt", truncation=True, max_length=max_length)
            t0 = time.perf_counter_ns()
            _ = cpu_model(**enc)
            lats.append(time.perf_counter_ns() - t0)
    del cpu_model
    return float(np.mean(lats)) / 1e6, float(np.percentile(lats, 95)) / 1e6

results_log = []
raw_predictions = {}

# --- Resume support: skip runs already written to RUN_LOG and reuse their preds CSV ---
_log_rows = []
if os.path.exists(RUN_LOG):
    with open(RUN_LOG, encoding="utf-8") as _f:
        for _line in _f:
            try:
                _log_rows.append(json.loads(_line))
            except Exception:
                pass
_done = {(r["task"], r["model"], r["seed"]) for r in _log_rows}
print(f"[resume] {len(_done)} run(s) already logged")

for task in BENCHMARK_TASKS:
    train_df, val_df, test_df, num_labels, source_repo = load_official_uit_dataset(task)
    raw_predictions[task] = {}

    for model_name in MODELS:
        seed_metrics = []
        raw_predictions[task][model_name] = {}

        for seed in SEEDS:
            if (task, model_name, seed) in _done:
                _p = f"Repro_Paper1_preds_{task}_{model_name.replace('/', '_')}_s{seed}.csv"
                if os.path.exists(_p):
                    raw_predictions[task][model_name][seed] = pd.read_csv(_p)["pred"].values
                    _rec = next((r for r in _log_rows
                                 if (r["task"], r["model"], r["seed"]) == (task, model_name, seed)),
                                None)
                    if _rec:
                        results_log.append(_rec)
                        seed_metrics.append((_rec["macro_f1"], _rec["weighted_f1"],
                                             _rec["accuracy"]))
                    print(f"[skip] {model_name} | seed {seed} (already done)")
                else:
                    print(f"[warn] logged but preds file missing: {model_name} s{seed} -> rerun")
                continue

            print(f"\n>>> Fine-tuning {model_name} on {task} (Seed: {seed}) <<<")
            set_deterministic_seed(seed)
            tokenizer = AutoTokenizer.from_pretrained(model_name)

            def tokenize_fn(batch, _tok=tokenizer):
                return _tok(batch["text"], truncation=True, max_length=128)

            tr_ds = Dataset.from_pandas(train_df).map(tokenize_fn, batched=True)
            va_ds = Dataset.from_pandas(val_df).map(tokenize_fn, batched=True)
            te_ds = Dataset.from_pandas(test_df).map(tokenize_fn, batched=True)

            model = AutoModelForSequenceClassification.from_pretrained(
                model_name, num_labels=num_labels
            )

            # Exact hyperparameters from ViSoBERT Section 4.1:
            # batch_size=40, max_seq_length=128, lr=2e-5, epochs=10, weight_decay=0.0
            train_args_kwargs = dict(
                output_dir=f"./checkpoints_{task}_{model_name.replace('/', '_')}_s{seed}",
                save_strategy="epoch",
                learning_rate=2e-5,
                per_device_train_batch_size=40,
                per_device_eval_batch_size=40,
                num_train_epochs=10,
                weight_decay=0.0,
                adam_epsilon=1e-8,
                warmup_ratio=0.1,
                load_best_model_at_end=True,
                metric_for_best_model="macro_f1",
                greater_is_better=True,
                fp16=torch.cuda.is_available(),
                logging_steps=50,
                report_to="none",
                save_total_limit=1,
                seed=seed,
            )
            train_args_kwargs[EVAL_STRATEGY_KEY] = "epoch"
            # transformers 5.x dropped several 4.x TrainingArguments fields (e.g. adam_epsilon).
            # Drop anything this installed version does not accept instead of dying mid-run.
            _supported = set(getattr(TrainingArguments, "__dataclass_fields__", {}))
            if _supported:
                _dropped = {k: v for k, v in train_args_kwargs.items() if k not in _supported}
                if _dropped:
                    print(f"[compat] transformers {__import__('transformers').__version__} "
                          f"does not accept: {sorted(_dropped)} -> dropped")
                train_args_kwargs = {k: v for k, v in train_args_kwargs.items()
                                     if k in _supported}
            if "fp16" not in train_args_kwargs:
                print("[compat] fp16 not applied on this version -> training in fp32 "
                      "(slower; record this in the report)")
            training_args = TrainingArguments(**train_args_kwargs)

            trainer = build_trainer(
                model=model,
                args=training_args,
                train_dataset=tr_ds,
                eval_dataset=va_ds,
                tokenizer=tokenizer,
                data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
                compute_metrics=compute_metrics,
            )

            trainer.train()

            # Held-out test inference, exactly once, with the best dev checkpoint.
            test_preds_out = trainer.predict(te_ds)
            test_preds = np.argmax(test_preds_out.predictions, axis=-1)
            raw_predictions[task][model_name][seed] = test_preds

            mf1 = f1_score(test_df["label"], test_preds, average="macro") * 100.0
            wf1 = f1_score(test_df["label"], test_preds, average="weighted") * 100.0
            acc = accuracy_score(test_df["label"], test_preds) * 100.0
            seed_metrics.append((mf1, wf1, acc))

            n_params = sum(p.numel() for p in model.parameters())
            fp32_mb = n_params * 4 / (1024 ** 2)
            mean_ms, p95_ms = measure_cpu_latency(
                model, tokenizer, test_df["text"].tolist()
            )

            record = {
                "task": task,
                "source_repo": source_repo,
                "model": model_name,
                "seed": seed,
                "n_test": int(len(test_df)),
                "macro_f1": round(mf1, 4),
                "weighted_f1": round(wf1, 4),
                "accuracy": round(acc, 4),
                "params": int(n_params),
                "fp32_mib": round(fp32_mb, 2),
                "cpu_latency_mean_ms": round(mean_ms, 3),
                "cpu_latency_p95_ms": round(p95_ms, 3),
            }
            results_log.append(record)
            with open(RUN_LOG, "a", encoding="utf-8") as f:
                f.write(json.dumps(record, ensure_ascii=False) + "\n")

            print(f"[{model_name} | Seed {seed}] Test: MF1={mf1:.2f}%, WF1={wf1:.2f}%, "
                  f"Acc={acc:.2f}%, params={n_params:,} (~{fp32_mb:.1f} MiB FP32), "
                  f"CPU {mean_ms:.2f} ms/sample")

            # Store per-sample predictions for the paired tests.
            pd.DataFrame({
                "idx": np.arange(len(test_df)),
                "gold": test_df["label"].values,
                "pred": test_preds,
            }).to_csv(
                f"Repro_Paper1_preds_{task}_{model_name.replace('/', '_')}_s{seed}.csv",
                index=False,
            )

            # Free VRAM before the next run: 9 sequential fine-tunes otherwise fragment
            # the caching allocator and OOM around run 5-9.
            del model, trainer
            gc.collect()
            torch.cuda.empty_cache()

        m_vals = [m[0] for m in seed_metrics]
        target_mf1 = PUBLISHED_TABLE4[task][model_name]["MF1"]
        mean_mf1 = float(np.mean(m_vals))
        std_mf1 = float(np.std(m_vals, ddof=1)) if len(m_vals) > 1 else 0.0
        print(f"== {model_name} on {task}: {mean_mf1:.2f} +/- {std_mf1:.2f} MF1 "
              f"(paper {target_mf1:.2f}) ==")

df_res = pd.DataFrame(results_log)
df_res.to_csv("Repro_Paper1_ViSoBERT_EMNLP2023_Comparison.csv", index=False)
display(df_res)

In [ ]:
# Cell 4: Multi-Class Statistical Verification
# McNemar is for paired binary decisions. For the multi-class Macro-F1 difference on
# 7-class UIT-VSMEC we use a paired bootstrap over the test set, and for the
# "Kaggle vs paper target" comparison we use an equivalence bound (never McNemar).
from scipy import stats as sps

BOOT_B = 2000
EQUIVALENCE_MARGIN_PP = 2.0  # pre-registered, do not tune after seeing results

def paired_bootstrap_mf1_diff(y_true, preds_a, preds_b, n_bootstraps=BOOT_B, alpha=0.05, seed=42):
    rng = np.random.RandomState(seed)
    n = len(y_true)
    diffs = []
    for _ in range(n_bootstraps):
        idx = rng.choice(n, size=n, replace=True)
        if len(np.unique(y_true[idx])) < 2:
            continue
        f1_a = f1_score(y_true[idx], preds_a[idx], average="macro") * 100.0
        f1_b = f1_score(y_true[idx], preds_b[idx], average="macro") * 100.0
        diffs.append(f1_a - f1_b)
    diffs = np.asarray(diffs)
    low, high = np.percentile(diffs, [alpha / 2 * 100, (1 - alpha / 2) * 100])
    # Add-one bootstrap p-value: never report an impossible exact 0.0000.
    p_val = (np.sum(diffs <= 0.0) + 1) / (len(diffs) + 1)
    return {
        "mean_diff": float(diffs.mean()),
        "ci_95": (round(float(low), 2), round(float(high), 2)),
        "p_value": float(p_val),
        "p_value_floor": f"< {1 / (len(diffs) + 1):.4f}",
        "is_significant": bool(low > 0.0),
        "n_boot": int(len(diffs)),
    }

def bootstrap_ci_metric(y_true, y_pred, metric="macro_f1", n_bootstraps=BOOT_B, seed=42):
    rng = np.random.RandomState(seed)
    n = len(y_true)
    vals = []
    for _ in range(n_bootstraps):
        idx = rng.choice(n, size=n, replace=True)
        if len(np.unique(y_true[idx])) < 2:
            continue
        if metric == "macro_f1":
            vals.append(f1_score(y_true[idx], y_pred[idx], average="macro") * 100.0)
        elif metric == "accuracy":
            vals.append(accuracy_score(y_true[idx], y_pred[idx]) * 100.0)
    vals = np.asarray(vals)
    return (round(float(np.percentile(vals, 2.5)), 2),
            round(float(np.percentile(vals, 97.5)), 2))

def benjamini_hochberg(p_values, alpha=0.05):
    p = np.asarray(p_values, dtype=float)
    order = np.argsort(p)
    m = len(p)
    adjusted = np.empty(m)
    prev = 1.0
    for rank in range(m - 1, -1, -1):
        idx = order[rank]
        prev = min(prev, p[idx] * m / (rank + 1))
        adjusted[idx] = prev
    return adjusted, adjusted < alpha

def clopper_pearson(k, n, alpha=0.05):
    if n == 0:
        return (float("nan"), float("nan"))
    lo = 0.0 if k == 0 else sps.beta.ppf(alpha / 2, k, n - k + 1)
    hi = 1.0 if k == n else sps.beta.ppf(1 - alpha / 2, k + 1, n - k)
    return round(float(lo) * 100, 2), round(float(hi) * 100, 2)

summary_rows = []
pvals_for_bh = []

for task in BENCHMARK_TASKS:
    y_test = load_official_uit_dataset(task)[2]["label"].values
    if "uitnlp/visobert" not in raw_predictions.get(task, {}):
        continue
    p_vis = raw_predictions[task]["uitnlp/visobert"][42]
    for baseline in ["vinai/phobert-base", "xlm-roberta-base"]:
        if baseline not in raw_predictions[task]:
            continue
        p_base = raw_predictions[task][baseline][42]
        boot = paired_bootstrap_mf1_diff(y_test, p_vis, p_base)
        ci = bootstrap_ci_metric(y_test, p_vis)
        target = PUBLISHED_TABLE4[task]["uitnlp/visobert"]["MF1"]
        lower_bound = target - EQUIVALENCE_MARGIN_PP
        observed = f1_score(y_test, p_vis, average="macro") * 100.0
        pvals_for_bh.append(boot["p_value"])
        summary_rows.append({
            "task": task,
            "comparison": f"ViSoBERT vs {baseline}",
            "delta_mf1_pp": round(boot["mean_diff"], 2),
            "delta_ci95": str(boot["ci_95"]),
            "p_value": round(boot["p_value"], 4),
            "significant": boot["is_significant"],
            "visobert_obs_mf1": round(observed, 2),
            "visobert_boot_ci95": str(ci),
            "paper_target_mf1": target,
            "within_equivalence_margin": bool(ci[0] >= lower_bound),
            "rank_order_holds": bool(observed > (f1_score(y_test, p_base, average="macro") * 100.0)),
        })
        print(f"\n=== {task}: ViSoBERT vs {baseline} (Seed 42) ===")
        print(f"Delta Macro-F1 = {boot['mean_diff']:.2f} pp | 95% CI {boot['ci_95']} | "
              f"p = {boot['p_value']:.4f} | significant={boot['is_significant']}")
        print(f"ViSoBERT observed MF1 = {observed:.2f}% (bootstrap CI {ci}), "
              f"paper target = {target:.2f}%, equivalence floor = {lower_bound:.2f}")

if summary_rows:
    adj, reject = benjamini_hochberg(pvals_for_bh)
    for row, a, r in zip(summary_rows, adj, reject):
        row["p_bh_adjusted"] = round(float(a), 4)
        row["significant_bh"] = bool(r)
    df_stats = pd.DataFrame(summary_rows)
    df_stats.to_csv("Repro_Paper1_Statistical_Concordance.csv", index=False)
    display(df_stats)
    print("\nBH-corrected decisions are in `significant_bh`. Report both raw and adjusted p-values.")